# Regression analysis

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn import linear_model
from sklearn.metrics import r2_score
from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Import data

In [ ]:
# Columns
columns = ['Offer_Id', 
           'Price_num', 
           'Mileage_num', 
           'HP_num', 
           'Make', 
           'Fuel_Type', 
           'Transmission', 
           'Init_Regist_MY']

# Read the data
df_orig = pd.read_csv('autoscout24_data_prepared.csv', 
                      sep=",", 
                      encoding='utf-8',
                      usecols=columns)

# Rename variable 'web-scraper-order' to 'apmt_id'
df_orig = df_orig.rename(columns={'Mileage_num': 'Mileage',
                                  'HP_num': 'HP',
                                  'Price_num': 'Price'})

# Extract year
df_orig['Init_Regist_MY'] = df_orig['Init_Regist_MY'].astype(str)
df_orig['Year'] = df_orig['Init_Regist_MY'].str.split('.').str[1].astype(int)
df_orig.drop('Init_Regist_MY', axis=1, inplace=True)

# Remove missing values
df = df_orig.dropna()
df.head(5)

# Remove duplicates
df = df.drop_duplicates()
df.head(5)

# Remove some extreme prices; keep only fuel type 'Diesel' and 'Benzin'
df = df.loc[(df['Price'] >= 10000) & (df['Price'] <= 100000) & (df['Fuel_Type'].isin(['Diesel','Benzin']))]
df


## One-hot encoding of categorical variables

In [ ]:
# Perform one-hot encoding
one_hot_encoded = pd.get_dummies(df['Fuel_Type']).astype(int)

# Concatenate the new one-hot encoded columns with the original DataFrame
df = pd.concat([df, one_hot_encoded], axis=1)

# Show the new DataFrame with one-hot encoding
df

## Create train and test samples (train = 80%, test = 20% of the data)

In [ ]:
# Create train and test samples
X_train, X_test, y_train, y_test = train_test_split(df[['Mileage', 'HP', 'Year', 'Benzin']], 
                                                    df['Price'], 
                                                    test_size=0.20, 
                                                    random_state=42)
# Show X_train
print('X_train:')
print(X_train.head(), '\n')

# Show y_train
print('y_train:')
print(y_train.head())

## Multiple linear regression

In [ ]:
# Add constant to the model
X_train_const = sm.add_constant(X_train)

# Create and fir the multiple regression model
olsmod = sm.OLS(y_train, X_train_const).fit()

# Print full model output
print(olsmod.summary())

### Plot histogram of multiple linear regression residuals

In [ ]:
# Add constant to the test set
X_test_const = sm.add_constant(X_test)

# Calculate predictions on the test set
y_pred = olsmod.predict(X_test_const)

# Calculate residuals on the test set
residuals = y_test - y_pred

# Plot histogram of residuals (based on the test sample)
fig = plt.figure(figsize=(6,4))
n, bins, patches = plt.hist(x=residuals, 
                            bins=25, 
                            color='blue',
                            alpha=0.5)
plt.xlabel('residuals', fontsize=10, labelpad=10)
plt.ylabel('frequency', fontsize=10, labelpad=10)
plt.title('Histogram of multiple linear regression residuals', fontsize=12, pad=10)

# Show plot
plt.show()

In [ ]:
# Goodness-of-fit on the *test* set (the OLS summary above is on training data)
from sklearn.metrics import mean_squared_error

r2_lr_test = r2_score(y_test, y_pred)
rmse_lr_test = mean_squared_error(y_test, y_pred) ** 0.5

print(f'Linear regression - Test R-squared: {r2_lr_test:.4f}')
print(f'Linear regression - Test RMSE: {rmse_lr_test:.2f}')

**Residual plot observation (linear regression):** The residuals are centered close to zero (mean ≈ CHF 240) but are right-skewed (skew ≈ 1.4): most residuals fall within roughly ±25'000 CHF, but there's a long tail of large positive residuals up to about +77'000 CHF, meaning the linear model tends to strongly *underpredict* the price of some expensive cars.

## Random forest regression

### Fit Random forest regression model

In [ ]:
# Initialize the random forest regressor
reg_rf = RandomForestRegressor(n_estimators=500, 
                               max_depth=10, 
                               random_state=5)

# Fit the model
reg_rf.fit(X_train, y_train)

# Calculate coefficient of determination (R-squared) and RMSE based on the test set
y_pred_rf = reg_rf.predict(X_test)
r2 = r2_score(y_test, y_pred_rf)
rmse_rf_test = mean_squared_error(y_test, y_pred_rf) ** 0.5
print(f'Random forest - Test R-squared: {r2:.4f}')
print(f'Random forest - Test RMSE: {rmse_rf_test:.2f}')

### Plot histogram of random forest regression residuals

In [ ]:
# Plot histogram of residuals (based on the test set)
fig = plt.figure( figsize=(6,4))
n, bins, patches = plt.hist(x=reg_rf.predict(X_test) - y_test, 
                            bins=25, 
                            color='blue',
                            alpha=0.5
                   )

# Set labels
plt.xlabel('residuals', fontsize=10, labelpad=10)
plt.ylabel('frequency', fontsize=10, labelpad=10)
plt.title('Histogram of random forest regression residuals', fontsize=12, pad=10)

# Show plot
plt.show()

**Residual plot observation (random forest):** The residuals are again centered close to zero (mean ≈ CHF 492) but are left-skewed (skew ≈ -1.0), with a heavier tail of large *negative* residuals (down to about -65'000 CHF), meaning the random forest tends to strongly *overpredict* a handful of cars. Overall the spread is tighter than linear regression (std ≈ CHF 10'250 vs. ≈ CHF 12'650), consistent with its higher test R² (0.65 vs. 0.47) and lower RMSE.

### Show feature importance

In [ ]:
# Get feature names from X_train
cols = X_train.columns

# Derive feature importance from random forest
importances = reg_rf.feature_importances_
std = np.std([tree.feature_importances_ for tree in reg_rf.estimators_], axis=0)
indices = np.argsort(importances)[::-1]

# Print col-names and importances-values
print(cols[indices])
print(importances[indices])

# Barplot with feature importance
df_fi = pd.DataFrame({'features':cols,'importances': importances})
df_fi.sort_values('importances', inplace=True)
df_fi.plot(kind='barh', 
           y='importances', 
           x='features', 
           color='darkred', 
           figsize=(6,3))

# Show plot
plt.show()

In [ ]:
# Report the top feature explicitly
print(f'Top feature for this run: {cols[indices][0]} (importance = {importances[indices][0]:.4f})')

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')